# DualSentinel — Calibração do Threshold do Detector

**Objectivo:** justificar empiricamente a escolha de `--threshold = 0.6` no `pipeline.py`.

O DualSentinel é **zero-shot** — não treina modelos com labels. Mas o `detector_score ∈ [0,1]` precisa de um ponto de corte para decidir quais janelas escalam ao SLM/Judge. Este notebook:

1. Agrega `windows_scored.json` de **todas as corridas** em `results/`
2. Mantém apenas janelas com `label ∈ {0, 1}` (rotuladas)
3. Calcula curvas **ROC**, **Precision-Recall** e **F1 vs. threshold**
4. Reporta os thresholds óptimos por **três critérios** (max F1, max Youden's J, FPR ≤ 5%)
5. Compara com o default actual (`0.6`) e produz uma recomendação defensável

> Não é treino — é **calibração** de um hiper-parâmetro de decisão sobre as labels disponíveis (LMD-2023 + AtomicRedTeam quando presentes).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", font_scale=1.0)
except Exception:
    sns = None

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (12, 4.2),
                     "axes.titleweight": "bold"})

DEFAULT_THRESHOLD = 0.6  # valor actual em src/pipeline.py

NB_DIR = Path.cwd()
DS_ROOT = next((p.resolve() for p in [NB_DIR.parent, NB_DIR]
                if (p / "src" / "pipeline.py").exists()), NB_DIR.parent)
RESULTS = DS_ROOT / "results"
print(f"DualSentinel root : {DS_ROOT}")
print(f"results/          : {RESULTS}  ({RESULTS.exists()=})")

## 1. Recolher janelas rotuladas de todas as corridas

Cada subdirectoria de `results/` contém um `windows_scored.json`. Mantemos só janelas com `label ∈ {0, 1}` (ignoramos `-1 = unknown`).

In [ ]:
rows = []
runs_seen = []
for run_dir in sorted(p for p in RESULTS.iterdir() if p.is_dir()):
    ws = run_dir / "windows_scored.json"
    if not ws.exists():
        continue
    try:
        data = json.loads(ws.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        continue
    if not data:
        continue
    runs_seen.append(run_dir.name)
    for w in data:
        rows.append({
            "run": run_dir.name,
            "label": w.get("label", -1),
            "detector_score": w.get("detector_score"),
            "if_score":  w.get("if_score"),
            "gru_score": w.get("gru_score"),
        })

raw = pd.DataFrame(rows)
labelled = raw[(raw["label"].isin([0, 1])) & (raw["detector_score"].notna())].copy()

print(f"Corridas inspeccionadas : {len(runs_seen)}")
print(f"Janelas totais          : {len(raw)}")
print(f"Janelas rotuladas (0/1) : {len(labelled)}")
print(f"  ↳ positivas (label=1) : {int((labelled['label']==1).sum())}")
print(f"  ↳ negativas (label=0) : {int((labelled['label']==0).sum())}")
if labelled["label"].nunique() < 2:
    print("\n⚠️  Apenas uma classe presente — calibração não é possível.")
    print("   Corre o pipeline com um dataset que contenha amostras maliciosas (e.g. AtomicRedTeam).")

## 2. Distribuição dos scores por classe

Histograma sobreposto: queremos ver separação visual entre as duas distribuições. Quanto maior o "gap", mais robusto é o threshold.

In [ ]:
if labelled["label"].nunique() == 2:
    fig, ax = plt.subplots(figsize=(11, 4))
    bins = np.linspace(0, 1, 30)
    for lbl, color, name in [(0, "#4caf50", "Benigno (label=0)"),
                             (1, "#e53935", "Malicioso (label=1)")]:
        s = labelled.loc[labelled["label"] == lbl, "detector_score"]
        ax.hist(s, bins=bins, alpha=0.55, color=color, label=f"{name} — n={len(s)}",
                edgecolor="white")
    ax.axvline(DEFAULT_THRESHOLD, color="black", linestyle="--",
               label=f"threshold actual = {DEFAULT_THRESHOLD}")
    ax.set_xlabel("detector_score")
    ax.set_ylabel("Janelas")
    ax.set_title("Distribuição de detector_score por classe")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Estatísticas separadas
    stats = labelled.groupby("label")["detector_score"].agg(
        ["count", "mean", "std", "min",
         lambda s: s.quantile(0.25), "median",
         lambda s: s.quantile(0.75), "max"]
    )
    stats.columns = ["n", "mean", "std", "min", "q25", "median", "q75", "max"]
    display(stats.round(3))
else:
    print("Skip — apenas uma classe disponível.")

## 3. Sweep de thresholds — F1, Precision, Recall, FPR

Varremos `t ∈ [0, 1]` em 101 passos. Para cada `t` calculamos `pred = (score ≥ t)` e a confusion matrix.

In [ ]:
def sweep(y_true: np.ndarray, scores: np.ndarray, n: int = 101) -> pd.DataFrame:
    ts = np.linspace(0.0, 1.0, n)
    out = []
    P = int((y_true == 1).sum())
    N = int((y_true == 0).sum())
    for t in ts:
        pred = (scores >= t).astype(int)
        tp = int(((pred == 1) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        fn = int(((pred == 0) & (y_true == 1)).sum())
        tn = int(((pred == 0) & (y_true == 0)).sum())
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / P if P else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        fpr  = fp / N if N else 0.0
        tpr  = rec
        youden = tpr - fpr
        out.append({"threshold": t, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
                    "precision": prec, "recall": rec, "f1": f1,
                    "fpr": fpr, "tpr": tpr, "youden_j": youden})
    return pd.DataFrame(out)

if labelled["label"].nunique() == 2:
    sw = sweep(labelled["label"].to_numpy(),
               labelled["detector_score"].to_numpy())

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
    axes[0].plot(sw["threshold"], sw["precision"], label="Precision", color="tab:blue")
    axes[0].plot(sw["threshold"], sw["recall"],    label="Recall",    color="tab:orange")
    axes[0].plot(sw["threshold"], sw["f1"],        label="F1",        color="tab:red", linewidth=2)
    axes[0].axvline(DEFAULT_THRESHOLD, color="black", linestyle="--",
                    label=f"actual ({DEFAULT_THRESHOLD})")
    axes[0].set_xlabel("threshold")
    axes[0].set_ylabel("score")
    axes[0].set_title("Precision / Recall / F1 vs. threshold")
    axes[0].legend()
    axes[0].set_ylim(-0.02, 1.02)

    axes[1].plot(sw["threshold"], sw["fpr"], label="FPR (falsos positivos)", color="firebrick")
    axes[1].plot(sw["threshold"], sw["tpr"], label="TPR (recall)", color="seagreen")
    axes[1].plot(sw["threshold"], sw["youden_j"], label="Youden's J", color="purple", linewidth=2)
    axes[1].axhline(0.05, color="grey", linestyle=":", label="FPR = 5%")
    axes[1].axvline(DEFAULT_THRESHOLD, color="black", linestyle="--")
    axes[1].set_xlabel("threshold")
    axes[1].set_title("FPR / TPR / Youden's J vs. threshold")
    axes[1].legend()
    axes[1].set_ylim(-0.02, 1.02)
    plt.tight_layout()
    plt.show()
else:
    sw = pd.DataFrame()
    print("Skip — apenas uma classe disponível.")

## 4. Curvas ROC e Precision-Recall

ROC e PR são **independentes do threshold** — quantificam a separabilidade global do score. AUC > 0.85 é considerado bom em deteção de anomalias em logs.

In [ ]:
if labelled["label"].nunique() == 2:
    from sklearn.metrics import (roc_curve, precision_recall_curve,
                                 roc_auc_score, average_precision_score)
    y = labelled["label"].to_numpy()
    s = labelled["detector_score"].to_numpy()

    fpr, tpr, _   = roc_curve(y, s)
    prec, rec, _  = precision_recall_curve(y, s)
    roc_auc       = roc_auc_score(y, s)
    pr_auc        = average_precision_score(y, s)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].plot(fpr, tpr, color="tab:blue", linewidth=2,
                 label=f"ROC (AUC={roc_auc:.3f})")
    axes[0].plot([0, 1], [0, 1], color="grey", linestyle="--", label="aleatório")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("Curva ROC")
    axes[0].legend()

    base = float((y == 1).mean())
    axes[1].plot(rec, prec, color="tab:red", linewidth=2,
                 label=f"PR (AP={pr_auc:.3f})")
    axes[1].axhline(base, color="grey", linestyle="--",
                    label=f"baseline (prevalência={base:.3f})")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Curva Precision-Recall")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    print(f"ROC-AUC : {roc_auc:.3f}")
    print(f"PR-AUC  : {pr_auc:.3f}  (baseline = prevalência = {base:.3f})")
else:
    print("Skip — apenas uma classe disponível.")

## 5. Thresholds óptimos por critério

Três critérios complementares — usar consoante a prioridade operacional:

| Critério | Quando usar |
|---|---|
| **Max F1** | balanço precision/recall (default académico) |
| **Max Youden's J** (`TPR − FPR`) | maximiza separação ROC; bom default em saúde/cyber |
| **FPR ≤ 5 %** | restrição operacional: SOC não tolera mais de 5 % de falsos alertas |

In [ ]:
if not sw.empty:
    def _row_at(t: float, criterion: str) -> dict:
        # Encontra a linha do sweep mais próxima de t
        idx = (sw["threshold"] - t).abs().idxmin()
        r = sw.loc[idx]
        return {"Critério": criterion, "Threshold": round(float(r["threshold"]), 3),
                "Precision": round(float(r["precision"]), 3),
                "Recall":    round(float(r["recall"]), 3),
                "F1":        round(float(r["f1"]), 3),
                "FPR":       round(float(r["fpr"]), 3),
                "TP": int(r["tp"]), "FP": int(r["fp"]),
                "FN": int(r["fn"]), "TN": int(r["tn"])}

    t_f1     = float(sw.loc[sw["f1"].idxmax(),       "threshold"])
    t_youden = float(sw.loc[sw["youden_j"].idxmax(), "threshold"])
    eligible = sw[sw["fpr"] <= 0.05]
    if not eligible.empty:
        # com FPR <= 5%, escolher o menor threshold (maximiza recall)
        t_fpr5 = float(eligible.iloc[eligible["recall"].argmax()]["threshold"])
    else:
        t_fpr5 = 1.0  # nenhum threshold satisfaz a restrição

    rows_opt = [
        _row_at(DEFAULT_THRESHOLD, f"Actual ({DEFAULT_THRESHOLD})"),
        _row_at(t_f1,              "Max F1"),
        _row_at(t_youden,          "Max Youden's J"),
        _row_at(t_fpr5,            "FPR ≤ 5 % (max recall)"),
    ]
    opt = pd.DataFrame(rows_opt)
    display(opt.style.hide(axis="index"))
else:
    opt = pd.DataFrame()
    print("Skip — sweep vazio.")

## 6. Recomendação

Regra de decisão para a defesa:

- Se **`FPR @ threshold=0.6` ≤ 5 %** *e* **F1 ≥ 0.8 × F1_max** → manter `0.6` (estável, redondo, fácil de comunicar).
- Caso contrário → recomendar `t_youden` (compromisso teórico) ou `t_fpr5` (compromisso operacional).

A célula seguinte produz a frase exacta a usar como justificação no relatório.

In [ ]:
if not opt.empty:
    actual = opt.iloc[0]
    f1_max = float(opt["F1"].max())
    keep   = (actual["FPR"] <= 0.05) and (actual["F1"] >= 0.8 * f1_max) and f1_max > 0

    print("─" * 70)
    print(f"Threshold actual              : {DEFAULT_THRESHOLD}")
    print(f"  → F1={actual['F1']}   FPR={actual['FPR']}   "
          f"Precision={actual['Precision']}   Recall={actual['Recall']}")
    print(f"F1 máximo observado            : {f1_max}  "
          f"(@ threshold={t_f1:.2f})")
    print(f"Youden's J óptimo              : threshold={t_youden:.2f}")
    print(f"FPR ≤ 5% (max recall)          : threshold={t_fpr5:.2f}")
    print("─" * 70)

    if keep:
        print("\n✅ RECOMENDAÇÃO: manter threshold = 0.6")
        print(f"   Justificação: FPR ({actual['FPR']:.1%}) ≤ 5% e F1 "
              f"({actual['F1']:.3f}) ≥ 80% do F1 máximo ({f1_max:.3f}).")
    else:
        alt = t_youden if abs(t_youden - DEFAULT_THRESHOLD) < 0.2 else t_fpr5
        print(f"\n⚠️  RECOMENDAÇÃO: considerar threshold ≈ {alt:.2f}")
        print(f"   Razão: o threshold actual ({DEFAULT_THRESHOLD}) "
              f"sai do envelope (FPR≤5% AND F1≥0.8·F1_max).")
        print(f"   Para alterar: edita o default em src/pipeline.py "
              f"(typer.Option na função `cli`).")
else:
    print("Sem recomendação — calibração não foi possível.")